# Exercise 7: From the log to the lake

Every run of your job does three things: **read** the new records of the topic `logs`,
**write** them into the lake (Parquet files in `data/lake/`), **remember** where it stopped.

This notebook gives you the building blocks (`tools/lakelib.py`). You write one function,
`run_once()`. Save your final version of this notebook into your `solution/` folder.

In [1]:
import sys; sys.path.append("/home/jovyan/tools")
from lakelib import *

(installing confluent-kafka==2.11.0 ...)
(installing duckdb==1.5.6 ...)


## 1. The example: a job that copies everything

It reads the topic from the beginning, every time. Run the next two cells and look at the
checker's last line.

In [2]:
def run_once():
    new = read_new(None)          # None = from the beginning of the topic
    write_files(new, LAKE)        # straight into the lake

reset()                           # empty lake
run_once()
check()

reset: lake, staging and state deleted
read_new: 2,000,000 new records
write_files: 2,000,000 records -> /home/jovyan/data/lake/logs/run-20261123-120520-dd328c

Lake check — topic 'logs' (4 partitions)   2026-11-23 12:07:15
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 518,662)     518,662           0          0           0
        1           [0, 474,692)     474,692           0          0           0
        2           [0, 519,644)     519,644           0          0           0
        3           [0, 487,002)     487,002           0          0           0
-------------------------------------------------------------------------------
    total      2,000,000 records   2,000,000           0          0           0

files under /home/jovyan/data/lake: 4 data files, 111.6 MB, 27.90 MB pe

{'topic': 2000000,
 'lake': 2000000,
 'dup': 0,
 'miss': 0,
 'lag': 0,
 'files': 4,
 'mb': 111.581829,
 'ok': True,
 'problems': []}

In [3]:
run_once()                        # the next run
check()

read_new: 2,000,000 new records
write_files: 2,000,000 records -> /home/jovyan/data/lake/logs/run-20261123-120715-72d424

Lake check — topic 'logs' (4 partitions)   2026-11-23 12:07:58
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 518,662)   1,037,324     518,662          0           0
        1           [0, 474,692)     949,384     474,692          0           0
        2           [0, 519,644)   1,039,288     519,644          0           0
        3           [0, 487,002)     974,004     487,002          0           0
-------------------------------------------------------------------------------
    total      2,000,000 records   4,000,000   2,000,000          0           0

files under /home/jovyan/data/lake: 8 data files, 223.2 MB, 27.90 MB per file on average
check took 0.8 s

Exa

{'topic': 2000000,
 'lake': 4000000,
 'dup': 2000000,
 'miss': 0,
 'lag': 0,
 'files': 8,
 'mb': 223.163658,
 'ok': False,
 'problems': ['2,000,000 duplicates']}

## 2. The building blocks

| Block | What it does |
|---|---|
| `load_state()` | where the last run stopped, from a small state file (`None` if there is none) |
| `save_state(new.end)` | stores where this run stopped in the state file |
| `lake_position()` | where the last run stopped, read from the lake itself (`None` if the lake is empty) |
| `read_new(start)` | the records after `start`, up to the end of the topic *now* (`new.end`) |
| `write_files(new, folder)` | writes them as Parquet into a new run folder inside `LAKE` or `STAGING` |
| `publish(run)` | moves a finished run folder from `STAGING` into the lake: one rename, a single step |

`LAKE` is what readers see; `STAGING` is a folder next to it that nobody reads.
`new.df.show()` shows the records of a run as a Spark DataFrame.

## 3. Your architecture

Write `run_once()` for the architecture you chose (A, B or C, see the README).
The order of the calls *is* your architecture.

In [5]:
def run_once():
    # 1. where did the last run stop? (read from the lake itself)
    start = lake_position()
    # 2. read what is new
    new = read_new(start)
    # 3. write it: first into STAGING, then into the lake in one step
    run = write_files(new, STAGING)
    publish(run)
    # 4. remember where you stopped (if your architecture needs it)
    # not needed: the lake itself is the state

## 4. Check it

`check_all` empties the lake, then runs your job in five situations: a first run, new records,
nothing new, a run that stops **while** writing, a run that stops **after** writing.
It takes 2-4 minutes.

Architecture C: readers use a view, so check through it: `check_all(run_once, lake_sql=DEDUP_SQL)`.

In [6]:
check_all(run_once)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             2,000,000           0        0           0  33.6s  PASS
2 new records arrive, run again         2,100,000           0        0           0   6.1s  PASS
3 nothing new, run again                2,100,000           0        0           0   0.1s  PASS
4 run stops WHILE writing: right after  2,100,000           0        0     100,000         PASS
  ... then the next run                 2,200,000           0        0           0   5.6s  PASS
5 run stops AFTER writing: right after  2,200,000           0        0     100,000         PASS
  ... then the next run                 2,300,000           0        0           0   5.8s  PASS
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 2000000,
   'lake': 2000000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 111.581829,
   'ok': True,
   'problems': []},
  33.6330041885376),
 ('2 new records arrive, run again',
  {'topic': 2100000,
   'lake': 2100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 112.740315,
   'ok': True,
   'problems': []},
  6.062999963760376),
 ('3 nothing new, run again',
  {'topic': 2100000,
   'lake': 2100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 112.740315,
   'ok': True,
   'problems': []},
  0.1026298999786377),
 ('4 run stops WHILE writing: right after',
  {'topic': 2200000,
   'lake': 2100000,
   'dup': 0,
   'miss': 0,
   'lag': 100000,
   'files': 8,
   'mb': 112.740315,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 2200000,
   'lake': 2200000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 12,
   'mb': 113.846265,
   'ok': True,
   'problems': []},
  

In [7]:
def run_once_A():
    # architecture A, only for comparison: state file saved after writing
    start = load_state()
    new = read_new(start)
    write_files(new, LAKE)
    save_state(new.end)

check_all(run_once_A)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             2,300,000           0        0           0  44.3s  PASS
2 new records arrive, run again         2,400,000           0        0           0   7.7s  PASS
3 nothing new, run again                2,400,000           0        0           0   0.1s  PASS
4 run stops WHILE writing: right after  2,420,176           0        0      79,824         PASS
  ... then the next run                 2,520,176      20,176        0           0   6.6s  FAIL
5 run stops AFTER writing: right after  2,620,176      20,176        0           0         FAIL
  ... then the next run                 2,720,176     120,176        0           0   4.8s  FAIL
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 2300000,
   'lake': 2300000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 116.76516,
   'ok': True,
   'problems': []},
  44.32167077064514),
 ('2 new records arrive, run again',
  {'topic': 2400000,
   'lake': 2400000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 117.922446,
   'ok': True,
   'problems': []},
  7.734457015991211),
 ('3 nothing new, run again',
  {'topic': 2400000,
   'lake': 2400000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 117.922446,
   'ok': True,
   'problems': []},
  0.06238198280334473),
 ('4 run stops WHILE writing: right after',
  {'topic': 2500000,
   'lake': 2420176,
   'dup': 0,
   'miss': 0,
   'lag': 79824,
   'files': 9,
   'mb': 118.167035,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 2500000,
   'lake': 2520176,
   'dup': 20176,
   'miss': 0,
   'lag': 0,
   'files': 13,
   'mb': 119.322375,
   'ok': False,
   'problems': ['